# Unificación de las alineaciones (*depth charts*) — 2016 a la temporada en curso

## Qué responde este notebook

`nflreadpy` publica las alineaciones en dos esquemas incompatibles: hasta 2024 traen temporada y
semana; de 2025 en adelante solo la fecha y hora en que se capturó la lista (ver la Fuente 5 de
[`1.1_calidad_fuentes.ipynb`](1.1_calidad_fuentes.ipynb) y
[`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md)). Sin semana no se pueden cruzar con las
estadísticas por `(season, week, team, player_id)`, y `depth_team` quedaría vacío de 2025 en
adelante.

Este notebook:

1. Muestra el problema: el esquema nuevo no tiene semana, solo la fecha de captura.
2. Explica la lógica de unificación con un caso concreto.
3. Verifica con casos reales conocidos que el mapeo produce alineaciones correctas y mide la
   cobertura de `depth_team` resultante.

La función resultante, `data.cargar_depth_charts_unificado`, es la que usa
`features.construir_tabla_modelado`, así que el análisis y el modelado de los notebooks siguientes
ya trabajan con las alineaciones unificadas. Qué es una alineación y qué significan sus columnas
está en el [glosario](../../../docs/data/GLOSARIO_VARIABLES.md) (sección 1 y apéndice B.3).

In [1]:
import sys
sys.path.insert(0, "../src")
import data

import pandas as pd

pd.set_option("display.width", 160)

## 1. El problema: un esquema sin semana

`data.cargar_depth_charts_historico()` (hasta 2024) trae `season` y `week`. El esquema nuevo, que se
lee con `nfl.load_depth_charts()` a partir de 2025, solo trae `dt`: la fecha y hora en que se capturó
la alineación. Sin temporada ni semana no se puede cruzar con `data.cargar_stats_semanales()` por
`(season, week, team, player_id)`.

In [2]:
import nflreadpy as nfl

crudo_2025 = nfl.load_depth_charts([2025]).to_pandas()
print("columnas del esquema 2025+:", list(crudo_2025.columns))
print("¿tiene season o week?", any(c in crudo_2025.columns for c in ("season", "week")))
print(f"fechas de captura distintas en 2025: {crudo_2025['dt'].nunique()}")

columnas del esquema 2025+: ['dt', 'team', 'player_name', 'espn_id', 'gsis_id', 'pos_grp_id', 'pos_grp', 'pos_id', 'pos_name', 'pos_abb', 'pos_slot', 'pos_rank']
¿tiene season o week? False
fechas de captura distintas en 2025: 221


## 2. La solución, con un caso concreto

Una alineación capturada un día cualquiera describe la que el equipo prepara para su **próximo**
partido, no la del que ya jugó. El calendario incluye todos los partidos de la temporada, jugados o
no, así que cada captura se puede asignar al primer partido de su equipo en esa fecha o después.

Ejemplo: Arizona Cardinals (`ARI`), 2025. Su semana 8 fue de descanso (*bye*; ver el
[glosario](../../../docs/data/GLOSARIO_VARIABLES.md), sección 2): el calendario salta de la semana 7
(19 de octubre) a la 9 (3 de noviembre).

In [3]:
cal_ari = data.cargar_calendario([2025])
cal_ari = cal_ari[(cal_ari["home_team"] == "ARI") | (cal_ari["away_team"] == "ARI")]
cal_ari = cal_ari[["season", "week", "gameday", "home_team", "away_team"]].sort_values("week")
print("calendario de ARI, 2025:")
print(cal_ari.to_string(index=False))

dt_ari = crudo_2025[(crudo_2025["team"] == "ARI") & (crudo_2025["pos_abb"] == "WR")]["dt"]
dt_ari = pd.to_datetime(dt_ari).dt.tz_localize(None).drop_duplicates().sort_values()
en_ventana_bye = dt_ari[(dt_ari > "2025-10-19") & (dt_ari <= "2025-11-03")]
asignacion = pd.merge_asof(
    en_ventana_bye.to_frame("dt"), cal_ari[["gameday", "week"]].sort_values("gameday"),
    left_on="dt", right_on="gameday", direction="forward",
)
print(f"\ncapturas de ARI entre el partido de la semana 7 (19-oct) y el de la semana 9 (3-nov): {len(en_ventana_bye)}")
print("semana a la que se asigna cada una:", asignacion["week"].value_counts().to_dict())

calendario de ARI, 2025:
 season  week    gameday home_team away_team
   2025     1 2025-09-07        NO       ARI
   2025     2 2025-09-14       ARI       CAR
   2025     3 2025-09-21        SF       ARI
   2025     4 2025-09-25       ARI       SEA
   2025     5 2025-10-05       ARI       TEN
   2025     6 2025-10-12       IND       ARI
   2025     7 2025-10-19       ARI        GB
   2025     9 2025-11-03       DAL       ARI
   2025    10 2025-11-09       SEA       ARI
   2025    11 2025-11-16       ARI        SF
   2025    12 2025-11-23       ARI       JAX
   2025    13 2025-11-30        TB       ARI
   2025    14 2025-12-07       ARI        LA
   2025    15 2025-12-14       HOU       ARI
   2025    16 2025-12-21       ARI       ATL
   2025    17 2025-12-28       CIN       ARI
   2025    18 2026-01-04        LA       ARI

capturas de ARI entre el partido de la semana 7 (19-oct) y el de la semana 9 (3-nov): 14
semana a la que se asigna cada una: {9: 14}


Las 14 capturas de Arizona entre esos dos partidos caen durante el descanso, y todas se asignan a la
semana 9, el siguiente partido real: no se pierden ni se inventa una semana 8 que ese equipo no jugó.
La asignación se hace con `pd.merge_asof(..., direction="forward")` entre las capturas de cada equipo
y las fechas de sus partidos. Si varias capturas caen en la misma semana de un equipo, se usa la más
reciente, la más cercana al partido. En el esquema nuevo, `pos_rank` (un orden sin empates dentro del
equipo) cumple el papel de `depth_team`.

`data.cargar_depth_charts_unificado()` implementa esta lógica y la combina con
`cargar_depth_charts_historico()` (hasta 2024) en una sola tabla `(season, week, team, player_id,
depth_team)`. Las alineaciones históricas tienen además dos detalles que la función corrige: usan el
código de equipo de la época (`OAK` hasta 2019, `SD` en 2016) en lugar del actual, que es el que usan
las estadísticas (`LV`, `LAC`), y traen una semana de más marcada como temporada regular (la 18 en
2016-2020, temporadas de 17 semanas). Ver [`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md).

In [4]:
depth_unificado = data.cargar_depth_charts_unificado(range(2016, 2026))
print("shape unificado (2016-2025):", depth_unificado.shape)
print(depth_unificado.groupby("season")["week"].agg(["min", "max", "nunique"]))

duplicados = depth_unificado.duplicated(subset=["season", "week", "team", "player_id"], keep=False)
print(f"\nfilas duplicadas jugador-equipo-semana: {duplicados.sum()} de {len(depth_unificado)}")

shape unificado (2016-2025): (30765, 5)
        min   max  nunique
season                    
2016.0  1.0  17.0       17
2017.0  1.0  17.0       17
2018.0  1.0  17.0       17
2019.0  1.0  17.0       17
2020.0  1.0  17.0       17
2021.0  1.0  18.0       18
2022.0  1.0  18.0       18
2023.0  1.0  18.0       18
2024.0  1.0  18.0       18
2025.0  1.0  18.0       18

filas duplicadas jugador-equipo-semana: 0 de 30765


La tabla unificada tiene 30,765 filas: 17 semanas por temporada de 2016 a 2020 y 18 desde 2021, sin
filas duplicadas por jugador, equipo y semana.

## 3. Verificación con dos casos conocidos

Marvin Harrison Jr. (Arizona) y Ja'Marr Chase (Cincinnati) fueron los receptores principales de sus
equipos en 2025. Si el mapeo funciona, deberían aparecer con `depth_team` = 1 (titular, o «WR1») en
casi todas las semanas.

In [5]:
jugadores = data.cargar_jugadores()

def depth_team_2025(player_id):
    filas = depth_unificado[(depth_unificado["player_id"] == player_id) & (depth_unificado["season"] == 2025)]
    return filas.sort_values("week")[["season", "week", "team", "depth_team"]]

mhj_id = jugadores[jugadores["display_name"] == "Marvin Harrison Jr."]["gsis_id"].iloc[0]
chase_id = jugadores[jugadores["display_name"] == "Ja'Marr Chase"]["gsis_id"].iloc[0]

print("Marvin Harrison Jr. (ARI), depth_team por semana 2025:")
print(depth_team_2025(mhj_id).to_string(index=False))

print("\nJa'Marr Chase (CIN), depth_team por semana 2025:")
print(depth_team_2025(chase_id).to_string(index=False))

Marvin Harrison Jr. (ARI), depth_team por semana 2025:
 season  week team  depth_team
 2025.0   1.0  ARI           1
 2025.0   2.0  ARI           1
 2025.0   3.0  ARI           1
 2025.0   4.0  ARI           1
 2025.0   5.0  ARI           1
 2025.0   6.0  ARI           1
 2025.0   7.0  ARI           1
 2025.0   9.0  ARI           1
 2025.0  10.0  ARI           1
 2025.0  11.0  ARI           1
 2025.0  12.0  ARI           1
 2025.0  13.0  ARI           1
 2025.0  14.0  ARI           1
 2025.0  15.0  ARI           1
 2025.0  16.0  ARI           1
 2025.0  17.0  ARI           1
 2025.0  18.0  ARI           4

Ja'Marr Chase (CIN), depth_team por semana 2025:
 season  week team  depth_team
 2025.0   1.0  CIN           1
 2025.0   2.0  CIN           1
 2025.0   3.0  CIN           1
 2025.0   4.0  CIN           1
 2025.0   5.0  CIN           1
 2025.0   6.0  CIN           1
 2025.0   7.0  CIN           1
 2025.0   8.0  CIN           1
 2025.0   9.0  CIN           1
 2025.0  11.0  CIN         

Los dos aparecen con `depth_team` = 1 en todas las semanas que jugó su equipo, salvo Harrison Jr. en
la semana 18 (`depth_team` = 4). Ese valor es el de la última alineación publicada antes de ese
partido; con estos datos no se puede saber por qué el equipo lo bajó. Las semanas que faltan son las
de descanso de cada equipo: la 8 de Arizona y la 10 de Cincinnati.

## 4. Cobertura de `depth_team`

Qué fracción de las filas de WR de la tabla de estadísticas recibe un `depth_team` al cruzarse con la
tabla unificada, por temporada.

In [6]:
wr_stats = data.cargar_stats_semanales(range(2016, 2026))
wr_stats = wr_stats[wr_stats["position"] == "WR"]
cruce = wr_stats.merge(depth_unificado, on=["season", "week", "team", "player_id"], how="left")
cobertura = cruce.groupby("season")["depth_team"].apply(lambda s: s.notna().mean())
print("cobertura de depth_team sobre filas WR, por temporada:")
print((cobertura * 100).round(1).astype(str) + "%")

cobertura de depth_team sobre filas WR, por temporada:
season
2016    96.3%
2017    94.6%
2018    94.2%
2019    93.4%
2020    93.1%
2021    88.7%
2022    89.1%
2023    90.3%
2024    89.3%
2025    96.7%
Name: depth_team, dtype: object


Sin unificar, 2025 no tendría ninguna fila con `depth_team`, porque el esquema nuevo no se puede cruzar
(sección 1); con la unificación tiene 96.7%. En las temporadas anteriores la cobertura va de 88.7% a
96.3%. Los huecos que quedan son, sobre todo, de receptores que esa semana no aparecen en ninguna
alineación de su equipo; una parte menor aparece con otra etiqueta, como regresador de patadas o en
otra posición.

## Cierre

- `data.cargar_depth_charts_unificado()` combina los dos esquemas en una sola tabla
  `(season, week, team, player_id, depth_team)` de 2016 a la temporada en curso; es la que usa
  `features.construir_tabla_modelado`.
- Cada captura del esquema nuevo se asigna al siguiente partido de su equipo, así que las semanas de
  descanso no se pierden ni se inventan.
- El mapeo coincide con lo esperado en dos receptores principales (Harrison Jr. y Chase), y la
  cobertura de `depth_team` en 2025 (96.7%) es comparable a la de años anteriores (88.7% a 96.3%).

Anterior: [`1.2_construccion_datos.ipynb`](1.2_construccion_datos.ipynb) · Siguiente:
[`1.4_promedios_sin_fuga.ipynb`](1.4_promedios_sin_fuga.ipynb)